# 📌 Step1：导入依赖库，鸢尾花数据集

## 引导思考

sklearn 内置鸢尾花数据集，标签不是 0/1，是 `0,1,2` 三类。
思考：

1. 标签 y 是数字 0,1,2，模型输出是什么形式？
2. 鸢尾花特征一共 4 个，分别是什么？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

# 设置pandas显示优化
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth',35)

# 设置显示中文
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)      # 标签：0=setosa，1=versicolor，2=virginica
class_names = iris.target_names # ['setosa', 'versicolor', 'virginica']

print("特征名称：", iris.feature_names)
print("类别名称：", class_names)
print(f"特征矩阵: {X.shape}")
print(f"标签取值: {sorted(y.unique())}") # y.unique()：取出标签里所有不重复的值，得到 array([0,1,2])
print("\n前5行样本：")
print(X.head())
print("\n各类样本数量：")
# y.value_counts()：统计每个标签出现多少次（默认输出顺序：数量从大到小）；.sort_index()：按照标签编号从小到大重新排序
print(y.value_counts().sort_index())

# 📌 Step2：EDA 可视化

## 引导提问

数据集拥有 4 个特征，不方便直接四维画图。
我们选择最重要的两个特征：花瓣长度、花瓣宽度，绘制散点图，观察三类花能不能被区分开。
思考：Setosa（类别 0）从图上看，能不能和另外两类完全分开？

In [ ]:
plt.figure(figsize=(8,6))
# 取出两个花瓣特征
feature1 = "petal length (cm)"
feature2 = "petal width (cm)"

colors = ["red", "green", "blue"]
for label in [0, 1, 2]:
    mask = (y == label)
    plt.scatter(X.loc[mask, feature1], X.loc[mask, feature2], c=colors[label], label=class_names[label])

plt.xlabel(feature1)
plt.ylabel(feature2)
plt.title("鸢尾花：花瓣长度 & 花瓣宽度 分布")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

从图中可看出：Setosa（类别 0）能和另外两类完全分开。但是另外两种花有一小部分的交界。

# 📌 Step3 数据集划分：训练集 & 测试集

## 引导提问

原始数据集三类花朵各 50 条，类别均衡。
如果随机划分，有可能出现测试集某一类样本偏少的情况。
**如何保证训练集、测试集中三类花的占比和原始数据保持一致？**
答：`train_test_split` 的 `stratify=y`，分层抽样。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y # 分层抽样核心参数，根据标签 y 的分布进行划分，保证训练、测试集中 0/1/2 三类比例和原始数据集相同。
)

print(f"训练集 X_train: {X_train.shape}")
print(f"测试集 X_test: {X_test.shape}")
print("\n训练集各类数量：")
print(y_train.value_counts().sort_index())
print("\n测试集各类数量：")
print(y_test.value_counts().sort_index())

# 📌 Step4 搭建基础模型：多项式逻辑回归（多分类）

## 引导提问

二分类逻辑回归默认只能区分两类；现在 3 个鸢尾花品种。
`LogisticRegression` 依靠参数 `multi_class` 切换多分类模式：

- `multi_class="ovr"`：一对多，拆成 3 组二分类任务
- `multi_class="multinomial"`：多项式逻辑回归，使用 Softmax，直接做多分类，效果更好

额外知识点：逻辑回归属于线性模型，**必须标准化特征**！

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. 多项式逻辑回归
lr_multi = LogisticRegression(
    random_state=42,
    max_iter=200  # 增大迭代次数，保证收敛
)
lr_multi.fit(X_train_scaled, y_train)

# 3. 预测
y_pred_lr = lr_multi.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)

print("========== 多项式逻辑回归 ==========")
print(f"测试集准确率：{acc_lr:.4f}")

# 查看预测概率（多分类输出3个概率，总和=1）
proba = lr_multi.predict_proba(X_test_scaled[:5]) # 每条样本输出一组概率，对应类别 0、1、2，概率相加 = 1；predict() 自动选取概率最大的类别作为最终预测结果。
print("\n前5条样本预测概率 [类别0, 类别1, 类别2]：")
print(proba.round(3))
print("预测类别：", y_pred_lr[:5])
print("真实类别：", y_test.values[:5])

### 结果解读

测试集准确率 **0.9333**，30 条测试样本里错 2 条。
观察前 5 条预测：

- 类别 0 (setosa) 的预测概率无限接近 1，几乎不会认错；
- 类别 1、2 之间概率比较接近，误差就发生在这两类重叠区域，和我们之前散点图观察到的现象一致。

# 📌 Step5 引导提问

泰坦尼克二分类我们看懂了混淆矩阵；现在是**3 分类混淆矩阵**。
思考两个问题：

1. 混淆矩阵横轴、纵轴分别代表什么？
2. 多分类下精确率 Precision、召回率 Recall、F1 是对每一类单独计算的，你能理解含义吗？

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# 1. 混淆矩阵
cm = confusion_matrix(y_test, y_pred_lr)
print("========== 混淆矩阵 ==========")
print(cm)

# 绘制热力图混淆矩阵
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("预测类别")
plt.ylabel("真实类别")
plt.title("逻辑回归 混淆矩阵")
plt.show()

# 2. 分类报告：每一类的 Precision / Recall / F1
print("\n========== 分类评估报告 ==========")
print(classification_report(y_test, y_pred_lr, target_names=class_names))

### 结果分析

混淆矩阵清晰验证了我们之前的猜想：

1. **setosa**：10 条全部预测正确，Precision、Recall、F1 全部 = 1.0，不存在任何混淆；
2. 错误只出现在 `versicolor` 和 `virginica` 互相认错：
真实 versicolor 有 1 条被预测成 virginica；真实 virginica 有 1 条被预测成 versicolor，合计 2 个错误样本，对应总体准确率 0.9333。

补充两个报告关键名词（写报告必备）

- `macro avg`：各类指标直接算术平均，适合均衡数据集（鸢尾花正好均衡）
- `weighted avg`：按每类样本数量加权平均，类别不均衡场景优先使用

# 📌 Step6 引导提问

逻辑回归是**线性模型**，决策边界是直线。
随机森林属于树模型，可以学习非线性边界。
思考两个问题：

1. 多分类随机森林类名是什么？（对比二分类 RandomForestClassifier）
2. 随机森林需要标准化特征吗？

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 树模型不需要标准化！直接使用原始特征 X_train
rf_cls = RandomForestClassifier(n_estimators=100, random_state=42)
rf_cls.fit(X_train, y_train)

y_pred_rf = rf_cls.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print("========== 随机森林多分类评估 ==========")
print(f"测试集准确率：{acc_rf:.4f}")

# 混淆矩阵
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("\n混淆矩阵：")
print(cm_rf)

print("\n========== 分类评估报告 ==========")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

# 特征重要性
feat_import = pd.DataFrame({
    "feature":X.columns,
    "importance":rf_cls.feature_importances_
}).sort_values("importance", ascending=False)
print("\n特征重要性：")
print(feat_import)

### 结果分析

1. 现象：默认参数随机森林准确率 **0.90**，低于多项式逻辑回归 0.9333。原因：
   - 数据集体量极小（总共仅 150 条样本），随机森林容易轻微过拟合；
   - 同时默认超参数不一定适配小数据集，需要网格搜索优化。
2. 特征重要性印证我们 EDA 观察：
`petal width`、`petal length` 两者合计权重接近 0.87，花瓣特征是区分鸢尾花的核心；花萼宽度区分能力最弱。
1. 混淆矩阵错误集中依旧在 versicolor ↔ virginica，Setosa 依旧全部识别正确。

# 📌 Step7 引导提问

随机森林当前使用默认参数，我们用 `GridSearchCV` 网格搜索寻找最优超参数。
思考问题：

1. 分类任务网格搜索 `scoring` 参数填什么？
2. 小数据集，`max_depth` 不宜设置过大，为什么？

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [60, 100, 140],
    "max_depth": [3, 5, 8, None]
}

rf_base = RandomForestClassifier(random_state=42)
grid = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)
print("✅网格搜索最优参数：", grid.best_params_)
print(f"5折交叉验证最优准确率：{grid.best_score_:.4f}")

best_rf = grid.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
acc_best_rf = accuracy_score(y_test, y_pred_best_rf)

print("\n======== 调参后最优随机森林测试集评估 ========")
print(f"测试集准确率：{acc_best_rf:.4f}")
print(classification_report(y_test, y_pred_best_rf, target_names=class_names))

### 调参结果简要分析

最优参数：`max_depth=3，n_estimators=60`
测试准确率提升至 **0.9667**，现在仅存在 1 条错误样本。
限制树深度，抑制了默认随机森林在小数据集上的过拟合，泛化能力提升。

# 📌 Step8：构建指标汇总表 + 柱状对比图

汇总全部模型结果，每组一根柱子展示准确率，方便横向对比

In [ ]:
result_table = pd.DataFrame({
    "模型": ["多项式逻辑回归", "随机森林(默认参数)", "随机森林(网格调参)"],
    "准确率": [0.9333, 0.9000, 0.9667]
})

plt.figure(figsize=(10,6))
plt.bar(result_table["模型"], result_table["准确率"], color=["#60a5fa","#f87171","#34d399"])
plt.ylim(0.85, 1.0)
plt.ylabel("准确率 Accuracy")
plt.title("各模型准确率对比")
plt.grid(axis="y", alpha=0.3)
plt.show()

# 📌 Step9 三大项目横向总结思考题

我们完成三个项目：

1. 泰坦尼克：**二分类**
2. 加州房价：**回归**
3. 鸢尾花：**多分类**

|维度|	回归任务（加州房价）|	二分类任务（泰坦尼克）|	多分类任务（鸢尾花）|
|:---:|:---:|:---:|:---:|
|预测目标	|连续数值	|2 种离散标签	|≥3 种离散标签|
|典型模型	|LinearRegression、XGBRegressor	|LogisticRegression、RandomForestClassifier	|LogisticRegression(multinomial)、RandomForestClassifier|
|核心评估指标	|MAE、RMSE、R2	|准确率、AUC、Precision/Recall/F1	|准确率、多分类混淆矩阵、各类别 Precision/Recall/F1、macro avg|
|数据集划分	|train_test_split（无 stratify）	|推荐 stratify=y	|必须推荐 stratify=y|
|线性模型要求	|需要标准化	|需要标准化	|需要标准化|
|输出函数	|直接输出数值	|sigmoid，输出二分类概率	|softmax，输出多类别概率|

### 📃拓展思考预留：

逻辑回归可以输出概率`predict_proba`，随机森林同样支持，多分类场景下这个功能有什么业务价值？

#### 1. 基础作用回顾

- `predict()`：直接输出最终类别（概率最大那一类）
- `predict_proba()`：输出**每一类对应的置信概率**，所有类别概率之和 = 1

以鸢尾花 3 分类为例，一条样本输出 `[0.001, 0.940, 0.059]`，代表：
setosa=0.001，versicolor=0.940，virginica=0.059，预测类别为 versicolor。

#### 2. 四大实际业务价值（可以直接写进报告）

##### （1）区分「高置信预测」和「模糊预测」

模型不只是输出标签，还能告诉你**它到底有没有把握**。
比如：

- `[0.01,0.98,0.01]`：高度确信是 versicolor，可以自动处理；
- `[0.02,0.48,0.50]`：versicolor 和 virginica 概率接近，属于模糊样本。

业务策略：模糊样本不自动判定，交给人工复核。
放到鸢尾花场景：两类交界容易出现模糊样本，人工鉴定。

##### （2）支持拒绝推断 / 阈值策略

你可以自定义阈值，而不是永远无脑选概率最大类别。
举例：要求预测置信度必须 >0.8，低于阈值的样本不给出判定。

> 场景：花卉自动分拣，拿不准的花朵人工分拣，减少错分损失。

##### （3）风险排序、优先级划分

假设业务有 3 类病害分类，不同类别处置成本不同。
哪怕不是第一名，次高概率的类别也具备参考意义，辅助风险评估。

##### （4）模型不确定性分析

用来定位数据集难以区分的样本（versicolor、virginica 交界样本），指导后续采集更多边界样本做数据增强。

> 补充重点：
> 逻辑回归输出的概率具备良好概率解释性；随机森林输出的概率是**投票占比**，理论上不是严格意义的概率，但工程上统一当作置信度使用。